# SemEval 2022 Task 2: Multilingual Idiomaticity Detection - One-Shot Baseline

## Task Overview
- **Objective**: Binary classification of Multi-Word Expressions (MWEs) as idiomatic (0) or literal (1)
- **Languages**: English (EN) and Portuguese (PT)
- **Model**: bert-base-multilingual-cased (mBERT)
- **Setting**: One-shot (1 positive + 1 negative example of each MWE in training)

## Key Differences from Zero-Shot:
1. **Training Data**: Uses BOTH train_zero_shot.csv + train_one_shot.csv
2. **Context**: NO context (only Target sentence, exclude Previous/Next)
3. **Input Format**: Two sentences (sentence1=Target, sentence2=MWE)
4. **Hyperparameters**: Same as zero-shot (matches official baseline)

## Reproducibility
- Fixed seed: **0** (matches official baseline)
- All hyperparameters match official implementation

---
## 1. SETUP & DEPENDENCIES
Install required libraries for training and evaluation.

In [1]:
# Install dependencies (run once in Colab)
!pip install -q torch transformers pandas scikit-learn sentencepiece datasets
print("✅ Dependencies installed")

✅ Dependencies installed


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import os
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
from datasets import Dataset

# Set seed for reproducibility (CRITICAL: matches official baseline)
SEED = 0
set_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"✅ Libraries imported | Seed set to {SEED}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ Libraries imported | Seed set to 0
PyTorch version: 2.9.0+cu126
CUDA available: True


In [3]:
!unzip SubTaskA.zip

Archive:  SubTaskA.zip
   creating: SubTaskA/
   creating: SubTaskA/SubTaskA/
   creating: SubTaskA/SubTaskA/Data/
  inflating: SubTaskA/SubTaskA/Data/dev.csv  
  inflating: SubTaskA/SubTaskA/Data/dev_gold.csv  
  inflating: SubTaskA/SubTaskA/Data/dev_submission_format.csv  
  inflating: SubTaskA/SubTaskA/Data/eval.csv  
  inflating: SubTaskA/SubTaskA/Data/eval_submission_format.csv  
  inflating: SubTaskA/SubTaskA/Data/train_one_shot.csv  
  inflating: SubTaskA/SubTaskA/Data/train_zero_shot.csv  
   creating: SubTaskA/SubTaskA/TestData/
  inflating: SubTaskA/SubTaskA/TestData/test.csv  
  inflating: SubTaskA/SubTaskA/TestData/test_submission_format.csv  
  inflating: SubTaskA/SubTaskA/TestData/train_one_shot.csv  


---
## 2. DATA LOADING & PREPROCESSING

### One-Shot Data Strategy:
- **Load BOTH** train_zero_shot.csv AND train_one_shot.csv
- **Combine** them into single training set
- **Format**: Two-sentence input
  - sentence1 = Target sentence (containing the MWE)
  - sentence2 = The MWE itself (explicit highlighting)

In [22]:
# ============================================================================
# CONFIGURATION - ABLATION STUDY SETTINGS
# ============================================================================

# ABLATION CONFIGURATION
CONTEXT_MODE = "WITH_CONTEXT"  # Options: "NO_CONTEXT" (baseline), "WITH_CONTEXT"
USE_SENTENCE2 = True         # Options: True (baseline with MWE), False (single sentence)

# Paths (hardcoded for Colab - change if running locally)
INPUT_DIR = Path("/content/SubTaskA/SubTaskA/Data")

# Output directory based on configuration
context_label = CONTEXT_MODE
sentence2_label = "WithMWE" if USE_SENTENCE2 else "NoMWE"
OUTPUT_DIR = Path(f"models/OneShot/{context_label}_{sentence2_label}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("="*70)
print("ONE-SHOT TRAINING - ABLATION STUDY")
print("="*70)
print(f"Context Mode: {CONTEXT_MODE}")
print(f"Use Sentence2 (MWE): {USE_SENTENCE2}")
print(f"Output Directory: {OUTPUT_DIR}")
print("="*70)

ONE-SHOT TRAINING - ABLATION STUDY
Context Mode: WITH_CONTEXT
Use Sentence2 (MWE): True
Output Directory: models/OneShot/WITH_CONTEXT_WithMWE


In [23]:
# ============================================================================
# DATA BUILDING FUNCTIONS (One-Shot Ablation Variants)
# ============================================================================

def build_no_context_with_mwe(row):
    """BASELINE: Target only + MWE as sentence2 (Official one-shot)"""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    mwe = str(row['MWE']).strip() if pd.notna(row['MWE']) else ""
    return target, mwe

def build_with_context_with_mwe(row):
    """ABLATION 1: Full context + MWE as sentence2"""
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    context = f"{prev} {target} {nxt}".strip()
    mwe = str(row['MWE']).strip() if pd.notna(row['MWE']) else ""
    return context, mwe

def build_no_context_no_mwe(row):
    """ABLATION 2: Target only, single sentence (no MWE separation)"""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    return target, None  # None indicates single-sentence format

def build_with_context_no_mwe(row):
    """ABLATION 3: Full context, single sentence (zero-shot format on one-shot data)"""
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    context = f"{prev} {target} {nxt}".strip()
    return context, None  # None indicates single-sentence format

# Select builder based on configuration
if CONTEXT_MODE == "NO_CONTEXT" and USE_SENTENCE2:
    build_function = build_no_context_with_mwe
    description = "BASELINE: Target only + MWE as sentence2"
elif CONTEXT_MODE == "WITH_CONTEXT" and USE_SENTENCE2:
    build_function = build_with_context_with_mwe
    description = "ABLATION: Full context + MWE as sentence2"
elif CONTEXT_MODE == "NO_CONTEXT" and not USE_SENTENCE2:
    build_function = build_no_context_no_mwe
    description = "ABLATION: Target only, single sentence"
else:  # WITH_CONTEXT and not USE_SENTENCE2
    build_function = build_with_context_no_mwe
    description = "ABLATION: Full context, single sentence (zero-shot format)"

print(f"✅ Selected builder: {description}")
print(f"   Context: {CONTEXT_MODE}")
print(f"   Sentence2 (MWE): {USE_SENTENCE2}")

✅ Selected builder: ABLATION: Full context + MWE as sentence2
   Context: WITH_CONTEXT
   Sentence2 (MWE): True


In [24]:
# ============================================================================
# LOAD AND COMBINE TRAINING DATA
# ============================================================================

print("\nLoading training datasets...")

# Load zero-shot training data
train_zero_df = pd.read_csv(INPUT_DIR / "train_zero_shot.csv")
print(f"  ✓ Zero-shot train: {len(train_zero_df)} samples")

# Load one-shot training data
train_one_df = pd.read_csv(INPUT_DIR / "train_one_shot.csv")
print(f"  ✓ One-shot train: {len(train_one_df)} samples")

# Combine both datasets (official one-shot uses BOTH)
train_combined = pd.concat([train_zero_df, train_one_df], ignore_index=True)
print(f"  ✓ Combined train: {len(train_combined)} samples")

# Build sentence pairs using the selected builder function
sentence_pairs = train_combined.apply(build_function, axis=1)

train_data = pd.DataFrame({
    "label": train_combined["Label"],
    "sentence1": [pair[0] for pair in sentence_pairs],  # Target sentence or context
    "sentence2": [pair[1] if pair[1] is not None else "" for pair in sentence_pairs]  # MWE or empty
})

print(f"\n✓ Training data prepared: {len(train_data)} samples")
print("  Format: label | sentence1 (Target) | sentence2 (MWE)")


Loading training datasets...
  ✓ Zero-shot train: 4491 samples
  ✓ One-shot train: 140 samples
  ✓ Combined train: 4631 samples

✓ Training data prepared: 4631 samples
  Format: label | sentence1 (Target) | sentence2 (MWE)


In [25]:
# ============================================================================
# LOAD DEV DATA (same format as training)
# ============================================================================

print("\nLoading dev dataset...")

# Load dev data
dev_df = pd.read_csv(INPUT_DIR / "dev.csv")
dev_gold = pd.read_csv(INPUT_DIR / "dev_gold.csv")
dev_df = dev_df.merge(dev_gold[["ID", "Label"]], on="ID", how="left")

# Build sentence pairs for dev using the same builder function
dev_sentence_pairs = dev_df.apply(build_function, axis=1)

dev_data = pd.DataFrame({
    "label": dev_df["Label"],
    "sentence1": [pair[0] for pair in dev_sentence_pairs],
    "sentence2": [pair[1] if pair[1] is not None else "" for pair in dev_sentence_pairs]
})

print(f"  ✓ Dev: {len(dev_data)} samples")


Loading dev dataset...
  ✓ Dev: 739 samples


In [26]:
# ============================================================================
# DATASET STATISTICS
# ============================================================================

print("\n" + "="*70)
print("DATASET STATISTICS")
print("="*70)

print(f"\nTraining Set:")
print(f"  Total: {len(train_data)} samples")
print(f"  Label 0 (Idiomatic): {(train_data['label']==0).sum()} ({(train_data['label']==0).sum()/len(train_data)*100:.1f}%)")
print(f"  Label 1 (Literal): {(train_data['label']==1).sum()} ({(train_data['label']==1).sum()/len(train_data)*100:.1f}%)")

print(f"\nDev Set:")
print(f"  Total: {len(dev_data)} samples")
print(f"  Label 0 (Idiomatic): {(dev_data['label']==0).sum()} ({(dev_data['label']==0).sum()/len(dev_data)*100:.1f}%)")
print(f"  Label 1 (Literal): {(dev_data['label']==1).sum()} ({(dev_data['label']==1).sum()/len(dev_data)*100:.1f}%)")

# Sample examples
print("\n" + "="*70)
print("SAMPLE TRAINING EXAMPLES (One-Shot Format)")
print("="*70)
for idx in [0, 1, 2]:
    print(f"\nExample {idx+1}:")
    print(f"  Label: {train_data.iloc[idx]['label']} ({'Idiomatic' if train_data.iloc[idx]['label']==0 else 'Literal'})")
    print(f"  Sentence1 (Target): {train_data.iloc[idx]['sentence1'][:80]}...")
    print(f"  Sentence2 (MWE): {train_data.iloc[idx]['sentence2']}")

print("\n✅ Data preprocessing complete!")


DATASET STATISTICS

Training Set:
  Total: 4631 samples
  Label 0 (Idiomatic): 2595 (56.0%)
  Label 1 (Literal): 2036 (44.0%)

Dev Set:
  Total: 739 samples
  Label 0 (Idiomatic): 336 (45.5%)
  Label 1 (Literal): 403 (54.5%)

SAMPLE TRAINING EXAMPLES (One-Shot Format)

Example 1:
  Label: 0 (Idiomatic)
  Sentence1 (Target): This inspired others to jump ropes as a leisure activity and the exercise was pa...
  Sentence2 (MWE): double dutch

Example 2:
  Label: 0 (Idiomatic)
  Sentence1 (Target): In the age of chivalry a man paid for the woman's dinner and to 'go Dutch' was t...
  Sentence2 (MWE): double dutch

Example 3:
  Label: 0 (Idiomatic)
  Sentence1 (Target): To her eternal credit, she kept both India and Pakistan as our friends during an...
  Sentence2 (MWE): double dutch

✅ Data preprocessing complete!


---
## 3. MODEL CONFIGURATION & TOKENIZATION

### Hyperparameters (FIXED - matches official baseline):
- Model: `bert-base-multilingual-cased`
- Learning rate: `2e-5`
- Batch size: `32`
- Epochs: `9`
- Max sequence length: `128`
- Seed: `0`

### Tokenization (Two-Sentence Format):
BERT will receive: `[CLS] sentence1 [SEP] sentence2 [SEP]`

In [27]:
# ============================================================================
# MODEL & TOKENIZER SETUP
# ============================================================================

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128

print(f"Loading model and tokenizer: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model (num_labels=2 for binary classification)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f"  ✓ Model loaded: {model.config.num_labels} classes")
print(f"  ✓ Tokenizer loaded: max_length={MAX_LENGTH}")

# Note: Warning about "classifier weights not initialized" is EXPECTED
# The classification head is randomly initialized and will be trained

Loading model and tokenizer: bert-base-multilingual-cased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✓ Model loaded: 2 classes
  ✓ Tokenizer loaded: max_length=128


In [28]:
# ============================================================================
# TOKENIZATION (TWO-SENTENCE FORMAT)
# ============================================================================

def tokenize_function_pair(examples):
    """
    Tokenize with TWO sentences:
    - sentence1: Target sentence
    - sentence2: MWE

    BERT sees: [CLS] sentence1 [SEP] sentence2 [SEP]
    token_type_ids: [0, 0, ..., 0, 1, 1, ..., 1]
    """
    return tokenizer(
        examples["sentence1"],  # First sentence
        examples["sentence2"],  # Second sentence (MWE)
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

print("Tokenizing datasets with two-sentence format...")

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_data)
dev_dataset = Dataset.from_pandas(dev_data)

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function_pair, batched=True)
dev_dataset = dev_dataset.map(tokenize_function_pair, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])
dev_dataset.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

print(f"  ✓ Train dataset tokenized: {len(train_dataset)} samples")
print(f"  ✓ Dev dataset tokenized: {len(dev_dataset)} samples")
print(f"  ✓ Format: Two sentences with token_type_ids")

Tokenizing datasets with two-sentence format...


Map:   0%|          | 0/4631 [00:00<?, ? examples/s]

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

  ✓ Train dataset tokenized: 4631 samples
  ✓ Dev dataset tokenized: 739 samples
  ✓ Format: Two sentences with token_type_ids


---
## 4. TRAINING

Training with fixed hyperparameters matching the official baseline.
- Early stopping enabled based on dev set F1 score
- Best checkpoint saved automatically

In [29]:
# Metric computation function
def compute_metrics(eval_pred):
    """Calculate Macro F1 score (average of F1 for each class)"""
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"macro-F1": macro_f1}

# Training arguments
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Hyperparameters (Baseline replication hence fixed)
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=9,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro-F1",

    # Reproducibility
    seed=SEED,

    # Logging
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Training configuration:\n")
print(f"Model: {MODEL_NAME}")
print(f"Context mode: {CONTEXT_MODE}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Max length: {MAX_LENGTH}")
print(f"Seed: {SEED}")
print(f"Output dir: {OUTPUT_DIR}")


/tmp/ipython-input-646976741.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training configuration:

Model: bert-base-multilingual-cased
Context mode: WITH_CONTEXT
Learning rate: 2e-05
Batch size: 32
Epochs: 9
Max length: 128
Seed: 0
Output dir: models/OneShot/WITH_CONTEXT_WithMWE


In [30]:
# ============================================================================
# START TRAINING
# ============================================================================

print("\n🚀 Starting one-shot training...\n")

# Train model
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Best model checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 score: {trainer.state.best_metric:.4f}")


🚀 Starting one-shot training...



Epoch,Training Loss,Validation Loss,Macro-f1
1,0.305700,0.774756,0.738788
2,0.153900,0.698687,0.780669
3,0.087200,0.781241,0.822687
4,0.048100,0.748228,0.843467
5,0.019900,0.867769,0.835320
6,0.013800,0.978540,0.845625
7,0.006300,1.008873,0.842220
8,0.007400,0.989100,0.840890
9,0.000400,0.985841,0.847334



✅ Training complete!
Best model checkpoint: models/OneShot/WITH_CONTEXT_WithMWE/checkpoint-1305
Best F1 score: 0.8473


---
## 5. EVALUATION & DETAILED RESULTS

Comprehensive analysis with breakdowns by:
- Class (Idiomatic vs Literal)
- Language (EN vs PT)
- Combined metrics

In [31]:
# ============================================================================
# GENERATE PREDICTIONS ON DEV SET
# ============================================================================

print("Generating predictions on dev set...")

# Get predictions
predictions = trainer.predict(dev_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)

# Load language information for detailed analysis
dev_full = pd.read_csv(INPUT_DIR / "dev.csv")
dev_full = dev_full.merge(dev_gold[["ID", "Label"]], on="ID", how="left")

# Create results dataframe
results_df = pd.DataFrame({
    'ID': dev_full['ID'],
    'Language': dev_full['Language'],
    'Gold_Label': dev_full['Label'],
    'Predicted_Label': pred_labels
})

print(f"  ✓ Predictions generated: {len(results_df)} samples")

Generating predictions on dev set...


  ✓ Predictions generated: 739 samples


In [32]:
# ============================================================================
# COMPREHENSIVE EVALUATION - DETAILED BREAKDOWN
# ============================================================================

print("="*80)
print("EVALUATION RESULTS - ONE-SHOT SETTING")
print("="*80)

print(f"\nTotal samples: {len(results_df)}")
print(f"Languages: {sorted(results_df['Language'].unique())}")
print(f"Classes: 0 (Idiomatic), 1 (Literal)")

# ============================================================================
# TABLE 1: OVERALL RESULTS (ALL LANGUAGES COMBINED)
# ============================================================================
print("\n" + "="*80)
print("TABLE 1: OVERALL RESULTS (ALL LANGUAGES COMBINED)")
print("="*80)

print("\n" + classification_report(
    results_df['Gold_Label'],
    results_df['Predicted_Label'],
    target_names=['Idiomatic', 'Literal'],
    digits=4
))

# Calculate overall macro F1 for summary
macro_f1 = f1_score(results_df['Gold_Label'], results_df['Predicted_Label'], average='macro')

# ============================================================================
# TABLE 2: RESULTS BY LANGUAGE (EN, PT)
# ============================================================================
print("\n" + "="*80)
print("TABLE 2: RESULTS BY LANGUAGE")
print("="*80)

for lang in sorted(results_df['Language'].unique()):
    lang_data = results_df[results_df['Language'] == lang]

    print(f"\n--- {lang} ({len(lang_data)} samples) ---")
    print(classification_report(
        lang_data['Gold_Label'],
        lang_data['Predicted_Label'],
        target_names=['Idiomatic', 'Literal'],
        digits=4
    ))

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "="*80)
print("SUMMARY - KEY METRICS")
print("="*80)

# Calculate macro F1 for each language
en_data = results_df[results_df['Language'] == 'EN']
pt_data = results_df[results_df['Language'] == 'PT']

en_macro_f1 = f1_score(en_data['Gold_Label'], en_data['Predicted_Label'], average='macro')
pt_macro_f1 = f1_score(pt_data['Gold_Label'], pt_data['Predicted_Label'], average='macro')

summary_data = [
    {'Metric': 'EN Macro F1', 'Value': f"{en_macro_f1:.4f}"},
    {'Metric': 'PT Macro F1', 'Value': f"{pt_macro_f1:.4f}"},
    {'Metric': 'Combined Macro F1', 'Value': f"{macro_f1:.4f}"}
]
summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE!")
print("="*80)
print(f"""
KEY TAKEAWAYS:
- Setting: One-Shot
- Input Format: Two sentences (Target + MWE)
- Overall Macro F1 (EN+PT): {macro_f1:.4f}
- EN Performance: {en_macro_f1:.4f}
- PT Performance: {pt_macro_f1:.4f}
""")

EVALUATION RESULTS - ONE-SHOT SETTING

Total samples: 739
Languages: ['EN', 'PT']
Classes: 0 (Idiomatic), 1 (Literal)

TABLE 1: OVERALL RESULTS (ALL LANGUAGES COMBINED)

              precision    recall  f1-score   support

   Idiomatic     0.8294    0.8393    0.8343       336
     Literal     0.8647    0.8561    0.8603       403

    accuracy                         0.8484       739
   macro avg     0.8470    0.8477    0.8473       739
weighted avg     0.8486    0.8484    0.8485       739


TABLE 2: RESULTS BY LANGUAGE

--- EN (466 samples) ---
              precision    recall  f1-score   support

   Idiomatic     0.8268    0.8132    0.8199       182
     Literal     0.8815    0.8908    0.8862       284

    accuracy                         0.8605       466
   macro avg     0.8542    0.8520    0.8531       466
weighted avg     0.8602    0.8605    0.8603       466


--- PT (273 samples) ---
              precision    recall  f1-score   support

   Idiomatic     0.8323    0.8701    0.

---
## 7. ABLATION STUDIES - SYSTEMATIC EXPERIMENTATION

### Instructions for Running Ablations:

To test different configurations:

1. **Go back to: "CONFIGURATION - ABLATION STUDY SETTINGS"**
2. **Change settings:**
   - `CONTEXT_MODE`: "NO_CONTEXT" or "WITH_CONTEXT"
   - `USE_SENTENCE2`: True or False

3. **Re-run all cells from Section 2 onwards**

4. **Record results in the table below**

### Expected Ablation Results Template:

| Context | Sentence2 | EN F1 | PT F1 | Combined F1 | Notes |
|---------|-----------|-------|-------|-------------|-------|
| NO_CONTEXT | True (MWE) | ?.???? | ?.???? | ?.???? | **BASELINE** - Official one-shot |
| WITH_CONTEXT | True (MWE) | ?.???? | ?.???? | ?.???? | Does context help with MWE? |
| NO_CONTEXT | False | ?.???? | ?.???? | ?.???? | Is MWE highlighting necessary? |
| WITH_CONTEXT | False | ?.???? | ?.???? | ?.???? | Zero-shot format on one-shot data |

### Research Questions:
- **Q1:** Does adding context improve one-shot performance?
- **Q2:** How much does explicitly highlighting the MWE (sentence2) help?
- **Q3:** Can zero-shot input format benefit from additional one-shot training data?
- **Q4:** Is there an interaction between context and MWE highlighting?

### Expected Patterns:
- Baseline (NO_CONTEXT + MWE): ~0.82 F1
- +Context might improve: better disambiguation
- -Sentence2 will likely drop: loses explicit MWE focus
- WITH_CONTEXT + No MWE: interesting hybrid approach

---
## 📝 NOTES

### One-Shot vs Zero-Shot Comparison:

| Aspect | Zero-Shot | One-Shot |
|--------|-----------|----------|
| Training Data | train_zero_shot.csv only | BOTH files |
| Context | ✅ Previous + Target + Next | ❌ Target only |
| Sentence2 | ❌ No | ✅ MWE |
| Input Format | Single sentence | Two sentences |
| token_type_ids | All 0s | 0s and 1s |
| Difficulty | 🔴 Harder | 🟡 Easier |

### Expected Performance:
- One-shot typically performs BETTER than zero-shot (model has seen examples of each MWE)
- Expected F1: ~0.80-0.85 (higher than zero-shot's ~0.75-0.80)

### Reproducibility Checklist:
- ✅ Seed fixed at 0 (matches official baseline)
- ✅ All hyperparameters match official implementation
- ✅ Same model: bert-base-multilingual-cased
- ✅ Same data: Both zero-shot + one-shot files
- ✅ Same input format: Two sentences
- ✅ Same evaluation metric: Macro F1

### For Research Paper:
```
We replicated the official one-shot baseline using bert-base-multilingual-cased
with fixed hyperparameters (lr=2e-5, batch=32, epochs=9, seed=0).
The model was trained on both zero-shot and one-shot data, using a two-sentence
input format (Target sentence + MWE) without surrounding context, and achieved
[X.XX] Macro F1 on the development set.
```